# Tests: `fastermodels.card` (source `nbs/02_card.ipynb`)

In [ ]:
from fastcore.test import *
from fastermodels.card import FORBIDDEN, check_card, render_card

In [ ]:
_meta = dict(
    name='test-resnet18-imagenette', base_model='torchvision/resnet18', license='bsd-3-clause',
    datasets=['frgfm/imagenette'], tags=['fasterai', 'pruning'],
    scope_line='Imagenette, n=3925; pipeline evidence, not a published claim.',
    input_shape='3x160x160',
    recipe={'prune': 'ratio 0.3, local, round_to 8', 'recovery': '3 epochs'},
    reference={'name': 'resnet18 fine-tuned on Imagenette', 'k': 3700, 'n': 3925, 'bytes': 44_726_568,
               'params': 11_181_642, 'macs': 1_824_000_000, 'peak_activation_bytes': 3_211_264},
    rows=[dict(artifact='pruned FP32', file='model.safetensors', params=8_900_000, bytes=35_600_000,
               macs=912_000_000, peak_activation_bytes=2_408_448,
               k=3680, n=3925, delta=-0.51, lo=-1.2, hi=0.2)],
    latency=None,
    provenance={'fasterai': '0.4.0', 'fastermodels': '0.1.0', 'torch': '2.9.1', 'measured_on': '2026-09-11'})

_card = render_card(_meta)

# a complete card says nothing a reader has to take on trust
test_eq(check_card(_card), [])

# the front matter the Hub reads
assert _card.startswith('---\n')
assert 'library_name: fastermodels' in _card
assert 'license: bsd-3-clause' in _card
assert 'base_model: torchvision/resnet18' in _card
assert '  - frgfm/imagenette' in _card

# the four criteria, each with the reference, the artifact and the gap between them, in units a reader reads
assert '| criterion | reference | this artifact | gap |' in _card
assert '| top-1 | 94.3 % | 93.8 % | -0.5 pt [-1.2, +0.2] |' in _card
assert '| size | 44.7 MB, 11.2 M params | 35.6 MB, 8.9 M params | -20.4 % |' in _card
assert '| memory | 3.2 MB | 2.4 MB | -25.0 % |' in _card

# half the reference's MACs reads -50.0 %, and never as a ratio the speedup rule would flag
assert '| MACs | 1.82 G | 912.0 M | -50.0 % |' in _card
assert '0.5x' not in _card and '2x' not in _card

# one line says what the brackets are, and no statistic is named
assert 'Gaps are measured on the same images as the reference; brackets give the 95 % interval.' in _card
for _word in ('Wilson', 'McNemar', 'bootstrap', 'CI'): assert _word not in _card, _word

# the resolution the memory was measured at is named
assert '3x160x160' in _card

# a latency that was not measured says so, in the language of the card, and never reads as a zero
assert 'not measured' in _card
assert 'non mesurée' not in _card
assert '0.00 ms' not in _card

# a reference value the producer did not measure reads n/a, and the card stays clean
_partial = render_card({**_meta, 'reference': {'name': 'source', 'k': 3700, 'n': 3925}})
assert '| MACs | n/a | 912.0 M | n/a |' in _partial
test_eq(check_card(_partial), [])

# the recipe is published only when the producer passes one
assert '## Recipe' in _card and '`prune`' in _card
_no_recipe = render_card({k: v for k, v in _meta.items() if k != 'recipe'})
assert '## Recipe' not in _no_recipe and 'round_to' not in _no_recipe
test_eq(check_card(_no_recipe), [])
test_eq(render_card({**_meta, 'recipe': {}}), _no_recipe)

# the publication checks are one line, without the evidence behind them
assert 'Publication checks' not in _card
_gate = [{'condition': i, 'name': n, 'passed': True} for i, n in enumerate(
    ['license', 'fresh-interpreter reload', 'parity', 'accuracy delta measured', 'exported file',
     'head and widths', 'size, memory, MACs and latency', 'card', 'clean-machine reload', 'on-target claim'])]
_all_pass = render_card({**_meta, 'gate': _gate})
assert _all_pass.rstrip().endswith(
    'Publication checks: 10/10 structural checks passed (they do not include the accuracy target, '
    'whose verdict is in each table above).'), _all_pass[-200:]
_one_failed = render_card({**_meta, 'gate': [{**_gate[0], 'passed': False}] + _gate[1:]})
assert _one_failed.rstrip().endswith(
    'Publication checks: 9/10 structural checks passed (they do not include the accuracy target, '
    'whose verdict is in each table above). Not passed: license.'), _one_failed[-200:]
for _c in (_all_pass, _one_failed): test_eq(check_card(_c), [])

# the accuracy target is reported under the table, and only when the row carries one
assert 'Accuracy target' not in _card
_met = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -1.71}]})
assert 'Accuracy target: -2.0 pt — met (lower bound -1.7)' in _met

# an interval that does not clear the target demonstrates nothing, which the card says as such
_straddles = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -2.14, 'hi': 0.2}]})
assert ('Accuracy target: -2.0 pt — not demonstrated (lower bound -2.1; the interval straddles the target)'
        in _straddles), _straddles
_below = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -4.0, 'hi': -2.6}]})
assert ('Accuracy target: -2.0 pt — not demonstrated (lower bound -4.0; the interval is entirely below '
        'the target)') in _below, _below
assert 'not met' not in _straddles and 'not met' not in _below
for _c in (_met, _straddles, _below): test_eq(check_card(_c), [])

# a row's notes are paragraphs under its table, after the target line, and a row without notes is unchanged
_noted = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -1.71,
                                         'notes': ['first.', 'second.']}]})
assert '\n\nfirst.\n\nsecond.\n' in _noted, _noted
assert _noted.index('first.') > _noted.index('Accuracy target'), _noted
test_eq(check_card(_noted), [])
test_eq(render_card({**_meta, 'rows': [{**_meta['rows'][0], 'notes': []}]}), _card)

# the ladder lets a reader pick another point on it; without one the card is unchanged
_ladder = render_card({**_meta, 'ladder': [
    {'name': 'pruned INT8', 'repo': 'FasterAI-Labs/resnet18-int8', 'delta': -1.4,
     'bytes': 11_200_000, 'peak_activation_bytes': 602_112, 'macs': 1_368_000_000},
    {'name': 'pruned 50 %', 'repo': 'FasterAI-Labs/resnet18-p50', 'delta': -3.1,
     'bytes': 22_300_000, 'peak_activation_bytes': 1_605_632, 'macs': 912_000_000}]})
assert '## Variants' in _ladder
assert '| variant | repo | top-1 gap (pt), worst published form | size | memory | MACs |' in _ladder
assert '| pruned INT8 | `FasterAI-Labs/resnet18-int8` | -1.4 | 11.2 MB | 0.6 MB | 1.37 G |' in _ladder
test_eq(check_card(_ladder), [])
assert '## Variants' not in _card
test_eq(render_card({**_meta, 'ladder': []}), _card)

# the Hub only accepts one of its own model ids in base_model, so a factory string goes to the provenance
_factory = render_card({**_meta, 'base_model': 'torchvision.models.resnet18 (IMAGENET1K_V1)'})
assert 'base_model:' not in _factory.split('---')[1]
assert _factory.split('## Provenance\n\n')[1].startswith('- Source model: torchvision.models.resnet18 (IMAGENET1K_V1)')
test_eq(check_card(_factory), [])

# a Hub id keeps the key, and nothing is repeated in the provenance
assert 'base_model: torchvision/resnet18' in _card.split('---')[1]
assert 'Source model' not in _card

# the Hub is only told a license once a person has validated it
_unvalidated = render_card({**_meta, 'license': {'id': 'bsd-3-clause', 'validated_by': ''}})
assert 'license:' not in _unvalidated.split('---')[1]
assert '- License: bsd-3-clause (not yet validated by a person)' in _unvalidated
test_eq(check_card(_unvalidated), [])

_validated = render_card({**_meta, 'license': {'id': 'bsd-3-clause', 'validated_by': 'the publisher'}})
assert 'license: bsd-3-clause' in _validated.split('---')[1]
assert 'not yet validated' not in _validated
test_eq(check_card(_validated), [])
test_eq(_validated, _card)   # a plain string is a license someone chose, and renders the same

# the card compares to something named, on a stated number of images, or it refuses to render
for _f in ('name', 'k', 'n'):
    with ExceptionExpected(KeyError, regex=_f):
        render_card({**_meta, 'reference': {k: v for k, v in _meta['reference'].items() if k != _f}})

# what the card does not show it ignores: parity is a gate matter, and so is the McNemar p
_old = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'agreement': 1.0, 'agreement_kind': 'same-precision',
                                       'p_mcnemar': 0.12}]})
test_eq(_old, _card)
assert 'agreement' not in _card

In [ ]:
# a measured latency is written with the device, the runtime, the precision and the batch
_measured = render_card({**_meta, 'latency': [dict(device='Jetson Orin NX', runtime='tensorrt', precision='fp16',
                                                   batch=1, median_ms=0.507, n_runs=100)]})
assert 'not measured' not in _measured
assert 'Jetson Orin NX' in _measured and 'tensorrt' in _measured and '0.507' in _measured
test_eq(check_card(_measured), [])

In [ ]:
# every forbidden phrase is reported, once, wherever it is injected
for _p in FORBIDDEN:
    test_eq(check_card(_card + f'\nThe artifact is {_p} on this line.\n'), [_p])
    test_eq(check_card(_card + f'\nThe artifact is {_p.upper()} on this line.\n'), [_p])

# whole words only: a phrase inside a longer word is not a claim
test_eq(check_card('the nanometre scale'), [])
test_eq(check_card('todolist'), [])

In [ ]:
# a speedup with neither device nor runtime on its line is reported
_flagged = check_card('the model is 2.3x faster')
test_eq(len(_flagged), 1)
assert '2.3x' in _flagged[0]
test_eq(len(check_card('the model is 2.3× faster')), 1)
test_eq(len(check_card('3x faster')), 1)

# a device alone, or a runtime alone, still leaves the claim unreadable
test_eq(len(check_card('the model is 2.3x faster on CPU')), 1)
test_eq(len(check_card('the model is 2.3x faster with onnxruntime')), 1)

# the same claim with both the device and the runtime it was measured on is not reported
test_eq(check_card('the model is 2.3x on CPU with onnxruntime'), [])
test_eq(check_card('2.3x vs the FP32 engine on a Jetson Orin NX with tensorrt, batch 1'), [])

# a number that is not a speedup is not a claim
test_eq(check_card('a 1x1 convolution'), [])
test_eq(check_card('the matrix is 3x4'), [])